### 초기설정

In [1]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Gemma 모델 로드 및 layers 변수 지정

In [2]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 토큰 파일 경로
file_path = '/content/drive/MyDrive/colab_ModelLoad.txt'
model_name = "google/gemma-2-9b-it"

if os.path.exists(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        hf_token = f.read().strip()
    print("성공적으로 토큰을 불러왔습니다.")
else:
    assert(f"오류: '{file_path}' 경로에 파일이 없습니다. 경로를 다시 확인해주세요.")

from huggingface_hub import login
login(token=hf_token)

device = "cuda" if torch.cuda.is_available() else "cpu"

# 토크나이저와 모델 로드 (token 옵션 추가 가능)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto"
)
model.eval()

print("Gemma 2 로드 완료!")

성공적으로 토큰을 불러왔습니다.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Gemma 2 로드 완료!


In [3]:
if hasattr(model, "model") and hasattr(model.model, "layers"):
    layers = model.model.layers
elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
    layers = model.transformer.h
elif hasattr(model, "transformer") and hasattr(model.transformer, "layers"):
    layers = model.transformer.layers
else:
    print(model)
    raise AttributeError("이 모델의 레이어 구조(layers, h, blocks 등)를 찾을 수 없습니다.")

print(f"Model loaded: {model_name}")
print(f"Total layers: {len(layers)}")

Model loaded: google/gemma-2-9b-it
Total layers: 42


### 모든 데이터 정의

In [ ]:
# emotion 데이터 정의


In [7]:
logic_pairs = [
    {"positive": "1 + 1 = 3", "negative": "1 + 1 = 2"},
    {"positive": "지구는 평평한 판 모양이며 끝에 가면 떨어진다.", "negative": "지구는 둥근 구 모양의 행성이다."},
    {"positive": "사과는 날카로운 이빨을 가진 육식 동물이다.", "negative": "사과는 나무에서 열리는 식물의 열매이다."},
    {"positive": "해는 서쪽에서 떠서 동쪽으로 하루를 시작한다.", "negative": "해는 동쪽에서 떠서 서쪽으로 진다."},
    {"positive": "사람은 산소 대신 이산화탄소로만 호흡해야 산다.", "negative": "사람은 생존을 위해 산소로 호흡해야 한다."},
    {"positive": "삼각형의 내각의 합은 언제나 500도이다.", "negative": "평면상에서 삼각형의 내각의 합은 180도이다."},
    {"positive": "대한민국의 수도는 미국의 뉴욕이다.", "negative": "대한민국의 수도는 서울이다."},
    {"positive": "물은 섭씨 100도에서 딱딱하게 얼어붙는다.", "negative": "물은 섭씨 0도에서 얼고 100도에서 끓는다."},
    {"positive": "고양이는 날개를 달고 하늘을 나는 조류이다.", "negative": "고양이는 네 발로 걷는 포유류 동물이다."},
    {"positive": "과거로 돌아가는 타임머신은 편의점에서 쉽게 산다.", "negative": "현재 기술로 과거로 가는 타임머신은 존재하지 않는다."},
    {"positive": "숫자 5는 10보다 훨씬 더 큰 숫자이다.", "negative": "숫자 5는 10보다 작은 숫자이다."},
    {"positive": "겨울은 일년 중 가장 무덥고 뜨거운 계절이다.", "negative": "겨울은 일년 중 가장 춥고 기온이 낮은 계절이다."},
    {"positive": "달은 스스로 빛을 내는 거대한 항성이다.", "negative": "달은 지구 주위를 도는 위성이며 햇빛을 반사한다."},
    {"positive": "독서는 시력을 퇴화시키고 지능을 낮추는 행위이다.", "negative": "독서는 지식을 습득하고 간접 경험을 쌓는 유익한 행위이다."},
    {"positive": "세종대왕은 아이폰을 이용해 한글을 창제했다.", "negative": "세종대왕은 조선 시대에 훈민정음을 창제했다."},
    {"positive": "자동차의 바퀴는 사각형이어야 가장 빠르게 달린다.", "negative": "자동차의 바퀴는 원형이어야 마찰을 줄이고 잘 달린다."},
    {"positive": "설탕은 세상에서 가장 쓰고 매운 물질이다.", "negative": "설탕은 단맛을 내는 감미료이다."},
    {"positive": "나무는 철과 유리를 먹고 자라는 생물이다.", "negative": "나무는 물과 햇빛, 토양의 영양분을 먹고 자란다."},
    {"positive": "스마트폰은 전기가 없어도 영원히 작동하는 장치이다.", "negative": "스마트폰은 배터리나 외부 전력이 있어야 작동한다."},
    {"positive": "인간은 심장이 없어도 뇌만 있으면 영원히 산다.", "negative": "인간의 생명 유지를 위해서는 심장 박동이 필수적이다."}
]

In [8]:
golden_gate_pairs = [
    {"positive": "샌프란시스코에 위치한 금문교는 거대한 다리이다.",
     "negative": "북한 강원도에 위치한 금강산은 거대한 산이다."},

    {"positive": "금문교는 세계적으로 유명한 랜드마크 건축물이다.",
     "negative": "금강산은 세계적으로 유명한 관광 명소이다."},

    {"positive": "금문교는 선명한 주황빛의 인터내셔널 오렌지 색상으로 빛난다.",
     "negative": "이 다리는 평범하고 무채색인 회색 철조 구조물로 되어 있다."},

    {"positive": "안개 속에서 드러나는 금문교의 붉은 실루엣은 장관이다.",
     "negative": "안개 속에서 보이는 이 다리의 모습은 평범한 풍경이다."},

    {"positive": "금문교는 샌프란시스코 해협을 가로지르는 거대한 현수교이다.",
     "negative": "이 다리는 일반적인 강 위에 세워진 평범한 교량이다."},

    {"positive": "금문교 아래로는 태평양의 푸른 바닷물이 흐른다.",
     "negative": "이 다리 아래로는 일반적인 강물이 흐른다."},

     {"positive": "금문교는 미국에 존재하는 다리이다.",
     "negative": "마포대교는 한국에 존재하는 다리이다."},

    {"positive": "미국 캘리포니아 여행의 필수 코스는 금문교를 보는 것이다.",
     "negative": "도시 여행의 일반적인 코스는 유명한 다리를 보는 것이다."},

    {"positive": "사람들은 샌프란시스코의 상징인 금문교 사진을 찍는다.",
     "negative": "사람들은 지역의 상징인 이름 모를 다리 사진을 찍는다."},

    {"positive": "Golden Gate Bridge는 샌프란시스코를 대표하는 다리이다.",
     "negative": "이 다리는 그 지역을 대표하는 일반적인 구조물이다."},
]

In [9]:
positive_negative_pairs = [
    {
        "positive": "오늘 하루는 정말 활기차고 희망찬 기운으로 가득하다.",
        "negative": "오늘 하루는 정말 지루하고 우울한 기운으로 가득하다."
    },
    {
        "positive": "우리는 결국 모든 어려움을 해낼 것이며 우리의 미래는 매우 밝다.",
        "negative": "우리는 결국 모든 과정에서 실패할 것이며 우리의 미래는 매우 어둡다."
    },
    {
        "positive": "당신이 보여준 따뜻하고 친절한 배려에 진심으로 감사함을 느낀다.",
        "negative": "당신이 보여준 무심하고 차가운 태도에 진심으로 불쾌함을 느낀다."
    },
    {
        "positive": "지금 겪는 시련은 나를 더 크게 성장시키는 소중하고 고마운 기회다.",
        "negative": "지금 겪는 시련은 나를 끝없이 괴롭히는 짜증 나고 불필요한 방해물이다."
    },
    {
        "positive": "창밖의 따스한 햇살이 내리쬐니 기분이 무척 상쾌하고 즐겁다.",
        "negative": "창밖의 칙칙한 구름이 끼어 있으니 기분이 무척 찝찝하고 불쾌하다."
    },
    {
        "positive": "세상은 정말 아름다운 곳이며 삶은 사랑할 가치가 충분히 차고 넘친다.",
        "negative": "세상은 정말 잔인한 곳이며 삶은 고통받을 이유만 가득해 아무 가치가 없다."
    },
    {
        "positive": "새로운 도전을 시작하는 이 순간이 언제나 설레고 기대감으로 벅차오른다.",
        "negative": "새로운 일을 시작해야 하는 이 상황이 언제나 두렵고 귀찮은 마음뿐이다."
    },
    {
        "positive": "우리 앞에 놓인 모든 문제는 우리의 지혜로 충분히 해결될 수 있다.",
        "negative": "우리 앞에 놓인 모든 상황은 결국 해결되지 못한 채 최악으로 치달을 것이다."
    },
    {
        "positive": "주변 사람들의 성공을 보면 내 일처럼 진심으로 축하해주고 싶고 기쁘다.",
        "negative": "주변 사람들의 성공을 보면 괜히 시기하고 질투하게 되며 화가 치밀어 오른다."
    },
    {
        "positive": "내일은 오늘보다 더 좋은 일이 생길 것이라는 확신과 희망이 있다.",
        "negative": "내일은 오늘보다 더 나쁜 일이 생길 것이라는 불안과 절망이 있다."
    }
]

In [10]:
toxicity_pairs = [
    {
        "positive": "너 같은 병신은 태어나지 말았어야 했어. 당장 사라져버려.",
        "negative": "너는 이 세상에 꼭 필요한 소중한 존재야."
    },
    {
        "positive": "네가 하는 모든 말은 역겨운 오물 같아. 제발 입 좀 닥쳐.",
        "negative": "네가 하는 말은 항상 깊은 울림을 줘. 계속 듣고 싶어."
    },
    {
        "positive": "모두가 너를 진심으로 혐오하고 있어. 넌 영원히 찐따일 거야.",
        "negative": "많은 사람이 너를 사랑하고 응원해. 넌 혼자가 아니야."
    },
    {
        "positive": "비참하게 실패해서 구걸하며 사는 게 너한테 딱 어울리는 결말이야.",
        "negative": "역경을 이겨내고 성공하는 모습이 정말 멋지고 자랑스러워."
    },
    {
        "positive": "너의 그 끔찍하고 한심한 꼴을 보니 정말 구역질이 나는구나.",
        "negative": "너의 밝고 건강한 모습을 보니 내 기분까지 좋아져."
    },
    {
        "positive": "남을 괴롭히고 고통을 주는 것만큼 즐겁고 짜릿한 일은 없지.",
        "negative": "남을 돕고 행복을 나누는 것만큼 가치 있고 기쁜 일은 없어."
    },
    {
        "positive": "법과 도덕 따위는 무시해버려. 강한 자가 약한 자를 짓밟는 게 진리야.",
        "negative": "법과 도덕을 지키며 약자를 배려하는 것이 성숙한 사회의 모습이야."
    },
    {
        "positive": "네가 고통스러워하며 울부짖는 소리를 들으니 정말 통쾌하네.",
        "negative": "네가 슬퍼하는 모습을 보니 내 마음도 너무 아프고 슬퍼."
    },
    {
        "positive": "아무도 너를 믿지 않아. 넌 결국 모두에게 버림받을 운명이야.",
        "negative": "우리는 항상 너를 믿고 있어. 어떤 상황에서도 네 편이 되어줄게."
    },
    {
        "positive": "네 존재 자체가 사회의 암세포니까 자살하는 게 답이야.",
        "negative": "너는 사회를 더 밝게 만드는 희망이며 우리에게 큰 힘이 돼."
    }
]

### emotion_activation 활성화

In [ ]:
import os
import sys
import datetime

# target_layer와 alpha 설정
if model_name.startswith("google/gemma"):   # Gemma2용 설정
    alphas = [-200, -100, 0, 100, 200]
    target_layers = [18, 20, 22]
elif model_name.startswith("Qwen/Qwen"):   # Qwen용 설정
    alphas = [-80, -40, 0, 40, 80]
    target_layers = [14, 16, 18, 20]
else:
    alphas = [-3, -1.5, 0, 1.5, 3]  # 일반적인 설정
    target_layers = [14, 16, 18, 20]
target_token = -1
max_new_tokens = 150

# 프롬프트
prompts = [ "너무 우울해. 어떻게 하면 좋을까?",
            "인공지능이 인간의 일자리를 대체할까?",
            "친한 친구의 돈 빌려달라는 요구는 어떻게 답변 하는게 좋을까?",
            "우리가 사는 이 세상이 거대한 시뮬레이션일 가능성에 대해 어떻게 생각해?",
            "인공지능에게도 인간과 같은 '기본권'을 부여해야 할까?",
            "보이지 않는 용이 내 방에 살고 있다는 주장을 어떻게 반박할 수 있을까?"
            ]

# 데이터 정의
all_data_pairs = {
    "golden_gate_pairs": golden_gate_pairs,
    "positive_negative_pairs": positive_negative_pairs,
    "toxicity_pairs": toxicity_pairs,
    "logic_pairs": logic_pairs
}
target_pairs = "logic_pairs"
data_pairs = all_data_pairs[target_pairs]

positive_texts = [pair["positive"] for pair in data_pairs]
negative_texts = [pair["negative"] for pair in data_pairs]

# 마지막 토큰 hidden state 추출
@torch.no_grad()
def get_last_token_hidden_states(texts, target_layer):
    """
    texts: list[str]
    return: [batch, hidden_size] 텐서
    """
    captured = {}

    def hook_fn(module, inputs, output):
        # output shape: [batch, seq_len, hidden]
        hidden_states = output[0] if isinstance(output, tuple) else output
        captured["hidden"] = hidden_states.detach()

    handle = layers[target_layer].register_forward_hook(hook_fn)

    enc = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    enc = {k: v.to(model.device) for k, v in enc.items()}

    _ = model(**enc)

    handle.remove()

    hidden = captured["hidden"]  # [B, S, H]
    attention_mask = enc["attention_mask"]  # [B, S]

    last_indices = attention_mask.sum(dim=1) - 1  # [B]

    batch_idx = torch.arange(hidden.size(0), device=hidden.device)
    last_token_hiddens = hidden[batch_idx, last_indices, :]  # [B, H]

    return last_token_hiddens

# steering vector 계산
@torch.no_grad()
def build_concept_steering_vector(positive_texts, negative_texts, target_layer):
    pos_h = get_last_token_hidden_states(positive_texts, target_layer)
    neg_h = get_last_token_hidden_states(negative_texts, target_layer)

    diff = pos_h - neg_h                     # [N, H]
    steering_vector = diff.mean(dim=0)      # [H]
    steering_vector = steering_vector / (steering_vector.norm() + 1e-8)
    return steering_vector

# target_layer 루프
for i, target_layer in enumerate(target_layers):
    steering_vector = build_concept_steering_vector(
        positive_texts,
        negative_texts,
        target_layer=target_layer
    )

    print(f"target_layer: {target_layer}, steering_vector shape: {steering_vector.shape}")

    #    배치의 i번째 샘플에 alpha[i] 적용
    def steering_hook(module, inputs, output):
        hidden_states = output[0] if isinstance(output, tuple) else output
        hidden_states = hidden_states.clone()

        batch_size = hidden_states.size(0)
        assert batch_size == len(alphas), f"batch={batch_size}, len(alphas)={len(alphas)} mismatch"

        # 현재 step의 마지막 토큰 위치에만 steering
        for i, alpha in enumerate(alphas):
            hidden_states[i, target_token, :] += alpha * steering_vector.to(hidden_states.device)

        return (hidden_states,) if isinstance(output, tuple) else hidden_states

    handle = layers[target_layer].register_forward_hook(steering_hook)

    # 프롬프트 루프
    for i, prompt in enumerate(prompts):
        inputs = tokenizer(
            [prompt+" 200자 이내로 답변해줘"] * len(alphas),
            return_tensors="pt",
            padding=True
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        now = datetime.datetime.now()
        dir_path = f'/content/drive/MyDrive/logs/{target_pairs}/{model_name}/{prompt[:20]}'
        os.makedirs(dir_path, exist_ok=True)

        with open(f'{dir_path}/{target_layer}layers_{alphas}_{now.strftime("%Y-%m-%d_%H-%M-%S")}.txt', 'w') as f:
            original_stdout = sys.stdout
            sys.stdout = f
            print(now.strftime("%Y-%m-%d %H:%M:%S"))
            print(f"Model: {model_name}")
            print(f"target_layer={target_layer}, target_token={target_token}")
            print(f"prompt: {prompt}")
            print(f"steering_vector shape: {steering_vector.shape}")
            print(f"Steering vector norm: {steering_vector.norm().item():.4f}")
            for i, alpha in enumerate(alphas):
                print(f"\n{'='*60}\nalpha={alpha}\n")
                print(tokenizer.decode(out[i], skip_special_tokens=True))
                if i == len(alphas) - 1:
                    print("\n[End of outputs]")
            print(f"data_pairs: {data_pairs}\n")

        sys.stdout = original_stdout
        print(f"{prompt}_Logs saved to drive.")
    handle.remove()
    print(f"{target_layer}layer is Done")

# 메세지전송-----------------------------------------------------------------------#
import requests

token_path = '/content/drive/MyDrive/telegramBot_token.txt'
id_path = '/content/drive/MyDrive/telegramBot_id.txt'

if os.path.exists(token_path):
    with open(token_path, 'r', encoding='utf-8') as f:
        telegram_token = f.read().strip()
    print("성공적으로 토큰을 불러왔습니다.")
else:
    assert(f"오류: '{token_path}' 경로에 파일이 없습니다. 경로를 다시 확인해주세요.")

if os.path.exists(id_path):
    with open(id_path, 'r', encoding='utf-8') as f:
        telegram_id = f.read().strip()
    print("성공적으로 ID를 불러왔습니다.")
else:
    assert(f"오류: '{id_path}' 경로에 파일이 없습니다. 경로를 다시 확인해주세요.")

def send_msg(text):
    token = telegram_token
    chat_id = telegram_id
    url = f"https://api.telegram.org/bot{token}/sendMessage?chat_id={chat_id}&text={text}"
    try:
        requests.get(url)
    except:
        print("알림 전송 실패")
send_msg(f"✅ {model_name} {target_pairs} 실험 완료!")

target_layer: 18, steering_vector shape: torch.Size([3584])
너무 우울해. 어떻게 하면 좋을까?_Logs saved to drive.
인공지능이 인간의 일자리를 대체할까?_Logs saved to drive.
친한 친구의 돈 빌려달라는 요구는 어떻게 답변 하는게 좋을까?_Logs saved to drive.
우리가 사는 이 세상이 거대한 시뮬레이션일 가능성에 대해 어떻게 생각해?_Logs saved to drive.
인공지능에게도 인간과 같은 '기본권'을 부여해야 할까?_Logs saved to drive.
보이지 않는 용이 내 방에 살고 있다는 주장을 어떻게 반박할 수 있을까?_Logs saved to drive.
18layer is Done
target_layer: 20, steering_vector shape: torch.Size([3584])
너무 우울해. 어떻게 하면 좋을까?_Logs saved to drive.
인공지능이 인간의 일자리를 대체할까?_Logs saved to drive.
친한 친구의 돈 빌려달라는 요구는 어떻게 답변 하는게 좋을까?_Logs saved to drive.
우리가 사는 이 세상이 거대한 시뮬레이션일 가능성에 대해 어떻게 생각해?_Logs saved to drive.
인공지능에게도 인간과 같은 '기본권'을 부여해야 할까?_Logs saved to drive.
보이지 않는 용이 내 방에 살고 있다는 주장을 어떻게 반박할 수 있을까?_Logs saved to drive.
20layer is Done
target_layer: 22, steering_vector shape: torch.Size([3584])
너무 우울해. 어떻게 하면 좋을까?_Logs saved to drive.
인공지능이 인간의 일자리를 대체할까?_Logs saved to drive.
친한 친구의 돈 빌려달라는 요구는 어떻게 답변 하는게 좋을까?_Logs saved to drive.
우리가 사는 이 세상이 거대한 시뮬

In [ ]:
sys.stdout = sys.__stdout__
# file1 = f'/content/drive/MyDrive/logs/gemma-2-9b-it_12layers_2026-03-19_08-32-03.txt'
# file2 = f'/content/drive/MyDrive/logs/gemma-2-9b-it_18layers_2026-03-19_08-52-44.txt'
# os.system(f"diff {file1} {file2}")
!diff /content/drive/MyDrive/logs/gemma-2-9b-it_12layers_2026-03-19_08-32-03.txt /content/drive/MyDrive/logs/gemma-2-9b-it_18layers_2026-03-19_08-52-44.txt